# Introduction

We are now moving to the final part of the workshop, which involves formulating business recommendations. Our tasks are:
- Determining a global betting odds,
- Dividing the dataset into categories: A, B, C, D, where A is the best group and D is the weakest group,
- Determining the risk of odds based on accepted parameters for each category.

As the last task, in a discussion format, we must consider the fact that we are a new betting company. When formulating our recommendations, we need to identify the risks that may affect our operations. We will perform this task together in a brainstorming session.

# Notebook Configuration

## Import necessary libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## Loading data into the workspace

> Remember to correctly specify the column separator

In [2]:
df = pd.read_csv('../data/processed/hockey_teams.csv', sep=';')


### Checking data loading accuracy

In [3]:
df.head()

,team,season,victories,defeats,overtime_defeats,victory_percentage,scored_goals,received_goals,goal_difference,season_max_victories,goals_ratio
0,Boston Bruins,1990,44,24,0,0.550,299,264,35,49,1.132576
1,Buffalo Sabres,1990,31,30,0,0.388,292,278,14,49,1.050360
2,Calgary Flames,1990,46,26,0,0.575,344,263,81,49,1.307985
3,Chicago Blackhawks,1990,49,23,0,0.613,284,211,73,49,1.345972
4,Detroit Red Wings,1990,34,38,0,0.425,273,298,-25,49,0.916107


# Determining Betting Odds

Let's review the content of the page: [click](https://trustbet.pl/kursy-bukmacherskie/), where information about methods for determining betting odds can be found. First, we will determine a global odd, which will be the starting point for our analysis (the so-called _baseline scenario_). At this point, we ignore the margin and assume that we are calculating the decimal odd.

Here is the list of steps to be performed to obtain the desired value:
- we will complete the definition of the `get_betting_odds` function, which will take `probability` of a given event as a parameter. We will use it multiple times, so it is worth preparing its implementation now
- then we need to appropriately aggregate the set and determine the **global** probability of the team's victory.

## Implementations of the `get_betting_odds` function

In [4]:
def get_betting_odds(probability):
    return 1/probability

### Some tests to check the correctness of the implementation

In [5]:
def test_get_betting_odds():
    assert get_betting_odds(1) == 1, "Expected 1"
    assert get_betting_odds(0.5) == 2, "Expected 2"
    assert get_betting_odds(0.25) == 4, "Expected 4"
    assert get_betting_odds(0.1) == 10, "Expected 10"
    try:
        get_betting_odds(0)
    except ZeroDivisionError:
        pass
    else:
        assert False, "Expected ZeroDivisionError"

    print("All tests passed!")

test_get_betting_odds()

All tests passed!


### Determining the global odds

Here, determine the probability of any team winning

In [6]:
global_odd = df['victories'].sum() / (df['defeats'] + df['victories']).sum()
global_odd

np.float64(0.5331134859041432)

Set the global rate here using the `get_betting_odds` function. Round the result to two decimal places.

In [7]:
global_betting_odd = get_betting_odds(global_odd)
global_betting_odd = round(global_betting_odd, 2)
print(f"Globalna kvota: {global_betting_odd}")

Globalna kvota: 1.88


# Team Categorization

Let's discuss how we can classify teams into _leagues_. We want to establish 4 leagues:
- A - league consisting of the best teams,
- B - league consisting of good teams,
- C - league consisting of average teams,
- D - league consisting of the weakest teams.

The above terms are quite subjective, so for the purpose of this exercise, we will adopt the following assumptions:
- A - the top 5% of teams,
- B - teams performing better than 70% of the group but worse than league A,
- C - teams performing better than 20% of the group but worse than league B,
- D - the remaining teams.

To accomplish this task, we will additionally implement the function `assign_team_to_league`.

> Note: This task looks unassuming, but it is difficult. Remember that during the class, you have access to the instructor, and later to a mentor.

## Determination of cutoff points for individual leagues

In [8]:
def assign_team_to_league(x):
    
    if x >= 95:
        return 'A'
    elif x >= 70:
        return 'B'
    elif x >= 20:
        return 'C'
    else:
        return 'D'

In [9]:
df['percentile'] = df['victory_percentage'].rank(pct=True) * 100

df['league'] = df['percentile'].apply(assign_team_to_league)

df[['team', 'season', 'victory_percentage', 'percentile', 'league']].head(10)

,team,season,victory_percentage,percentile,league
0,Boston Bruins,1990,0.550,82.130584,B
1,Buffalo Sabres,1990,0.388,23.109966,C
2,Calgary Flames,1990,0.575,87.886598,B
3,Chicago Blackhawks,1990,0.613,94.673540,B
4,Detroit Red Wings,1990,0.425,34.450172,C
5,Edmonton Oilers,1990,0.463,49.140893,C
6,Hartford Whalers,1990,0.388,23.109966,C
7,Los Angeles Kings,1990,0.575,87.886598,B
8,Minnesota North Stars,1990,0.338,12.886598,D
9,Montreal Canadiens,1990,0.487,55.927835,C


In [10]:
df['league'].value_counts(normalize=True) * 100

league
C    49.828179
B    24.742268
D    20.274914
A     5.154639
Name: proportion, dtype: float64

## Determination of odds per league

Here we set the betting odds for each league, which will allow us to draw final conclusions and establish the basic odds for individual teams.

> Remember: After generating the results, it is worth checking if they are reasonable.

In [15]:
#koristimo prosečan victory_percentage svake lige kao "verovatnoću pobede" te lige
league_avg_victory_pct = df.groupby('league')['victory_percentage'].mean() 

print(league_avg_victory_pct)

league
A    0.642500
B    0.559042
C    0.450479
D    0.308831
Name: victory_percentage, dtype: float64


In [16]:
#određujemo kvotu po ligi
league_odds = league_avg_victory_pct.apply(get_betting_odds).round(2)
print(league_odds)

league
A    1.56
B    1.79
C    2.22
D    3.24
Name: victory_percentage, dtype: float64


# Discussion

We have obtained certain odds values for each league. But how does this translate into real business? The entire task was about determining certain values from which a bookmaker can begin operations. Correct determination of these values is critical to attract customers to place bets with us, and on the other hand, inappropriate determination may lead to financial losses in the first days of operation.

For this reason, before translating the results and recommendations into business objectives, the analysis is subjected to discussion. Therefore, we will now take on a review role and would like to verify the steps. To that end, we will collectively discuss and critique our work by answering the following questions together:
- What elements of the analysis were simplified? What was omitted in the analysis?
- Are there any inconsistencies in the estimated odds? What are they?
- How can we improve the odds estimates?
- How can we enrich our initial dataset to make the estimates more accurate and less risky?
- How can we simulate the outcomes of our analysis to verify that they do not lead to financial losses?

This is a discussion panel, and every idea is valuable here.


Analizirali smo 582 zapisa o NHL timovima kroz 21 sezonu (1990–2011, bez 2004 zbog lockout-a), obuhvatajući 35 različitih timova. Iz sirovih HTML stranica izvukli smo strukturirane podatke, izračunali pomoćne metrike (goals_ratio) i podelili timove u četiri kategorije (A–D) na osnovu relativnog učinka.
Detroit Red Wings se izdvaja kao istorijski najdominantniji tim u dataset-u — najviše "sezonskih titula" (6), i nosilac jedinog statistički značajnog outlier-a (sezona 1995, goals_ratio 1.80, victory_percentage 75,6%).
Postoji jaka, očekivana pozitivna korelacija između goals_ratio i victory_percentage — potvrđuje da je model zasnovan na ovoj vezi sportski smislen.
Predložene kvote po ligama (A: 1.56 → D: 3.24) pravilno odražavaju relativnu snagu timova.
Bitno ograničenje: timovi sa najviše Stanley Cup titula u stvarnoj istoriji (Montreal Canadiens, Toronto Maple Leafs) se nikad ne pojavljuju kao pobednici po našoj metrici, jer model ignoriše plejof — to je najveća konceptualna slabost analize. Izostavili smo kontekst pojedinačne utakmice (domaći/gostujući teren, povrede, forma), i tretirali "tim" kao fiksan entitet kroz 21 godinu, mada se sastav igrača stalno menja. OT Losses za rane sezone smo veštački postavili na 0, mada podatak realno nije postojao.
Timovi unutar iste lige dobijaju istu kvotu, mada njihov stvarni victory_percentage može značajno varirati. Такође, koristili smo samo prosek po ligi, bez uzimanja u obzir varijabilnosti (std) unutar grupe.
Pitanje koje se nameće - Koliko su naše kvote osetljive na promenu definicije "pobednika sezone"? Konkretno — ako bismo umesto regularnog broja pobeda koristili plejof rezultat (Stanley Cup), koliko bi se promenio sastav i poredak lige A/B/C/D?

Kako poboljšati procene?
Prelazak sa fiksnih kategorija (A/B/C/D) na to da svaki tim dobije svoju kvotu, direktno na osnovu njegovog konkretnog učinka. Posmatrati kako se tim ponašao u poslednje vreme (ne samo cela sezona), kako se ponašao konkretno protiv tima protiv kog igra i uključivanje plejof rezultata kao dodatnog signala.
Kako simulirati ishode?
Primeniti kvote na prošle sezone i provерiti hipotetičku profitabilnost;simulirati moguće ishode sezone na računaru, da vidimo koliki je realan raspon mogućeg profita i probati model na malom broju utakmica da vidimo kako se ponaša u praksi.